In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pip install psycopg2-binary sqlalchemy

Read CSV File

In [ ]:
df = pd.read_csv('data/cleaned_data.csv')

In [ ]:
df.head()


Profitability Metric Calculation

In [ ]:
#profit per unit
df['profit_per_unit'] = df['gross_profit'] / df['units']

In [ ]:
# Profit Contribution
total_profit = df['gross_profit'].sum()

df['Total Profit Contribution %'] = (
    df['gross_profit'] / total_profit
) * 100

In [ ]:
# Revenue Contribution
total_sales = df['sales'].sum()

df['Revenue Contribution %'] = (
                                       df['sales'] / total_sales
                               ) * 100

In [ ]:
# Gross Margin
df['gross_margin %'] = (df['gross_profit'] / df['sales']) * 100

In [ ]:
df.head()

Margin Volitility

In [ ]:
#convert date time at dd-mm-yyy format
df['order_date'] = pd.to_datetime(
    df['order_date'],
    dayfirst=True
)

In [ ]:
# convert into datetime at yyy-mm-dd format
#df['order_date'] = pd.to_datetime(df['order_date'])


In [ ]:
#check datatype
print(df['order_date'].dtype)


In [ ]:
#create month column
df['Month'] = df['order_date'].dt.to_period('M')

In [ ]:
df.columns


In [ ]:
# Check result
print(df[['order_date', 'Month']].head())

In [ ]:
# Gross Margin
df['gross_margin %'] = (df['gross_profit'] / df['sales']) * 100

In [ ]:
#Monthly Margin
monthly_margin = df.groupby(
    ['product_name', 'Month']
)['gross_margin %'].mean().reset_index()

In [ ]:
#Volatility
margin_volatility = monthly_margin.groupby(
    'product_name'
)['gross_margin %'].std().reset_index()

In [ ]:
#Rename column
margin_volatility.columns = [
    'product_name',
    'margin_volatility'
]

In [ ]:
#Merge Back
df = df.merge(
    margin_volatility,
    on='product_name',
    how='left'
)

In [ ]:
df['gross_margin %'] = df['gross_margin %'].round(2)
df['margin_volatility'] = df['margin_volatility'].round(2)
df['profit_per_unit']=df['profit_per_unit'].round(2)

In [ ]:
df['Total Profit Contribution %']=df['Total Profit Contribution %'].round(2)
df['Revenue Contribution %']=df['Revenue Contribution %'].round(2)

Product-Level Profitability Analysis

In [ ]:
product_analysis = df.groupby('product_name').agg({
    'sales': 'sum',
    'gross_profit': 'sum',
    'gross_margin %': 'mean',
    'units': 'sum'
}).reset_index()

Rank Products by Gross Profit

In [ ]:
top_profit_products = product_analysis.sort_values(
    by='gross_profit',
    ascending=False
)
top_profit_products.head(10)

Rank Products by Gross Margin

In [ ]:
top_margin_products = product_analysis.sort_values(
    by='gross_margin %',
    ascending=False
)
top_margin_products.head(10)

High - Profit / High - Margin Products

In [ ]:
avg_profit = product_analysis['gross_profit'].mean()
avg_margin = product_analysis['gross_margin %'].mean()

high_profit_high_margin = product_analysis[
    (product_analysis['gross_profit'] > avg_profit) &
    (product_analysis['gross_margin %'] > avg_margin)
    ]
high_profit_high_margin

High - Sales / Low - Margin Products

In [ ]:
avg_sales = product_analysis['sales'].mean()

high_sales_low_margin = product_analysis[
    (product_analysis['sales'] > avg_sales) &
    (product_analysis['gross_margin %'] < avg_margin)
    ]
high_sales_low_margin

Low - Sales / Low - Profit Products

In [ ]:
low_sales_low_profit = product_analysis[
    (product_analysis['sales'] < avg_sales) &
    (product_analysis['gross_profit'] < avg_profit)
    ]
low_sales_low_profit

In [ ]:
df.head()

Division-Level Performance Analysis

In [ ]:
division_analysis = df.groupby('division').agg({
    'sales': 'sum',
    'gross_profit': 'sum',
    'gross_margin %': 'mean',
    'units': 'sum'
}).reset_index()

In [ ]:
division_analysis

Average margin by division

In [ ]:
#Rank Divisions by Margin
division_analysis.sort_values(
    by='gross_margin %',
    ascending=False
)

In [ ]:
#Revenue vs Profit Imbalance
#Create Profit-to-Sales Ratio
division_analysis['profit_ratio %'] = (
    division_analysis['gross_profit']
    / division_analysis['sales']
) * 100

In [ ]:
#Calculate Benchmarks
avg_div_profit = division_analysis['gross_profit'].mean()

avg_div_margin = division_analysis[
    'gross_margin %'
].mean()

In [ ]:
#Strong Division
strong_divisions = division_analysis[
    (division_analysis['gross_profit'] > avg_div_profit) &
    (division_analysis['gross_margin %'] > avg_div_margin)
]
strong_divisions

In [ ]:
#Margin-Issue Divisions
margin_issue_divisions = division_analysis[
    (division_analysis['sales']
     > division_analysis['sales'].mean()) &

    (division_analysis['gross_margin %']
     < avg_div_margin)
]

margin_issue_divisions

In [ ]:
#Create Division Performance Category
def division_performance(row):

    if (
        row['gross_profit'] > avg_div_profit and
        row['gross_margin %'] > avg_div_margin
    ):
        return 'Strong Financial Efficiency'

    elif (
        row['sales'] > division_analysis['sales'].mean() and
        row['gross_margin %'] < avg_div_margin
    ):
        return 'Structural Margin Issue'

    else:
        return 'Moderate Performance'


division_analysis['division_performance'] = (
    division_analysis.apply(
        division_performance,
        axis=1
    )
)

In [ ]:
#Final Output
division_analysis

Profit Concentration (Pareto) Analysis

In [ ]:
#Product-wise Revenue & Profit
pareto_df = df.groupby('product_name').agg({
    'sales': 'sum',
    'gross_profit': 'sum'
}).reset_index()

In [ ]:
#Sort by Revenue
pareto_df = pareto_df.sort_values(
    by='sales',
    ascending=False
)

In [ ]:
#Calculate Revenue Contribution %
total_sales = pareto_df['sales'].sum()

pareto_df['revenue_contribution %'] = (
    pareto_df['sales'] / total_sales
) * 100

In [ ]:
#Calculate Cumulative Revenue %
pareto_df['cumulative_revenue %'] = (
    pareto_df['revenue_contribution %']
).cumsum()

In [ ]:
#Identify Products Driving 80% Revenue
top_80_revenue_products = pareto_df[
    pareto_df['cumulative_revenue %'] <= 80
]
top_80_revenue_products

In [ ]:
#Profit Pareto Analysis
#Sort by Profit
profit_pareto = pareto_df.sort_values(
    by='gross_profit',
    ascending=False
)

In [ ]:
#Profit Contribution %
total_profit = profit_pareto['gross_profit'].sum()

profit_pareto['profit_contribution %'] = (
    profit_pareto['gross_profit']
    / total_profit
) * 100

In [ ]:
#Cumulative Profit %
profit_pareto['cumulative_profit %'] = (
    profit_pareto['profit_contribution %']
).cumsum()

In [ ]:
#Products Driving 80% Profit
top_80_profit_products = profit_pareto[
    profit_pareto['cumulative_profit %'] <= 80
]
top_80_profit_products

#Detect Over-Dependency Risk

In [ ]:
#Count Top Products
num_top_products = len(top_80_profit_products)

total_products = df['product_name'].nunique()

dependency_ratio = (
    num_top_products / total_products
) * 100

print(dependency_ratio)

Detect Congestion-Prone States/Regions

In [ ]:
#State-wise Revenue
state_sales = df.groupby('state/province').agg({
    'sales': 'sum',
    'gross_profit': 'sum'
}).reset_index()

In [ ]:
#Sort by Sales
state_sales = state_sales.sort_values(
    by='sales',
    ascending=False
)

In [ ]:
#Revenue Contribution %
total_state_sales = state_sales['sales'].sum()

state_sales['revenue_contribution %'] = (
    state_sales['sales']
    / total_state_sales
) * 100

In [ ]:
#Cumulative Contribution
state_sales['cumulative_revenue %'] = (
    state_sales['revenue_contribution %']
).cumsum()

Identify High Dependency States

In [ ]:
high_dependency_states = state_sales[
    state_sales['cumulative_revenue %'] <= 80
]
high_dependency_states

SQL Connection

In [ ]:
df.head()

Cost Structure Diagnostics

In [ ]:
#Create Cost Efficiency KPI
df['cost_efficiency_ratio %'] = (
    df['cost'] / df['sales']
) * 100

In [ ]:
#cost_efficiency_ratio % round 2 digit
df['cost_efficiency_ratio %'] = (
    df['cost_efficiency_ratio %']
    .round(2)
)

In [ ]:
#cost_efficiency_ratio % round 2 digit
df['cost_efficiency_ratio %'] = (
    df['cost_efficiency_ratio %']
    .round(2)
    .astype(int)
)

In [ ]:
#Product-wise Cost Analysis
cost_analysis = df.groupby('product_name').agg({
    'sales': 'sum',
    'cost': 'sum',
    'gross_profit': 'sum',
    'gross_margin %': 'mean',
    'cost_efficiency_ratio %': 'mean'
}).reset_index()

In [ ]:
#Cost vs Sales Scatter Analysis
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.scatter(
    cost_analysis['sales'],
    cost_analysis['cost']
)

plt.xlabel('sales')
plt.ylabel('cost')

plt.title('cost vs sales analysis')

plt.show()

In [ ]:
#Benchmarks
avg_cost_ratio = cost_analysis[
    'cost_efficiency_ratio %'
].mean()

avg_margin = cost_analysis[
    'gross_margin %'
].mean()

In [ ]:
#filter products
cost_heavy_margin_poor = cost_analysis[
    (cost_analysis['cost_efficiency_ratio %']
     > avg_cost_ratio) &

    (cost_analysis['gross_margin %']
     < avg_margin)
]

cost_heavy_margin_poor

In [ ]:
#Identify Pricing Inefficiencies
avg_sales = cost_analysis['sales'].mean()

pricing_inefficiencies = cost_analysis[
    (cost_analysis['sales'] > avg_sales) &

    (cost_analysis['gross_margin %']
     < avg_margin)
]

pricing_inefficiencies

In [ ]:
#Create Action Recommendation KPI
def recommendation(row):

    if (
        row['cost_efficiency_ratio %']
        > avg_cost_ratio
        and
        row['gross_margin %']
        < avg_margin
    ):
        return 'Cost Renegotiation Needed'

    elif (
        row['sales'] > avg_sales
        and
        row['gross_margin %']
        < avg_margin
    ):
        return 'Repricing Required'

    elif (
        row['sales'] < avg_sales
        and
        row['gross_profit']
        < cost_analysis['gross_profit'].mean()
    ):
        return 'Discontinuation Review'

    else:
        return 'Healthy Product'

    # Add new column into dataframe
cost_analysis['business_recommendation'] = (
    cost_analysis.apply(
        recommendation,
        axis=1
    )
)

In [ ]:
#Merge Into Main DataFrame
df = df.merge(
    cost_analysis[
        ['product_name', 'business_recommendation']
    ],
    on='product_name',
    how='left'
)

In [ ]:
#verify
df[
    [
        'product_name',
        'business_recommendation'
    ]
].head()


In [ ]:
#Final Output
cost_analysis.head()


In [ ]:
#Convert Month Column to String
df['Month'] = df['Month'].astype(str)

In [ ]:
df.columns

In [ ]:
df.head()


In [ ]:
def margin_risk(row):

    if row['gross_margin %'] < 15:
        return 'High Risk'

    elif row['gross_margin %'] < 30:
        return 'Medium Risk'

    else:
        return 'Low Risk'


df['margin_risk'] = (
    df.apply(
        margin_risk,
        axis=1
    )
)

In [ ]:
#Rename Columns Properly
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('%', 'percent')
)

In [ ]:
# final csv file
df.to_csv("final_nassau_data.csv", index=False)

In [ ]:
from sqlalchemy import create_engine
# Step 1: Connect to PostgreSQL
# Replace placeholders with your actual details
username = "postgres"      # default user
password = "pooja" # the password you set during installation
host = "localhost"         # if running locally
port = "5432"              # default PostgreSQL port
database = "Nassau_Canddy_DistributorDB"    # the database you created in pgAdmin
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")


In [ ]:
# Step 2: Load DataFrame into PostgreSQL
table_name = "sales_data"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)
print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")